In [ ]:
import cv2
import numpy as np
import time


cap = cv2.VideoCapture(0)


#  Enhancement: Choose your parameter by your self - Interactive
cv2.namedWindow("Controls")
cv2.createTrackbar("Param1", "Controls", 100, 300, lambda x: None)
cv2.createTrackbar("Param2", "Controls", 30, 100, lambda x: None)

# FPS information
start_time = time.time()
frame_count = 0

# ── Improvement 4: Stability – store the previously detected circle ──

prev_circle = None


while True:


    ret, frame = cap.read()
    if not ret:
        break


    frame_count += 1


    # Step 1: Convert frame to grayscale

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # Step 2: Apply median blur to reduce noise

    gray = cv2.medianBlur(gray, 5)


    # Apply histogram equalization to improve contrast

    gray = cv2.equalizeHist(gray)


    # IMPROVEMENT 2: Parameter Tuning via Trackbars

    param1 = cv2.getTrackbarPos("Param1", "Controls")
    param2 = cv2.getTrackbarPos("Param2", "Controls")


    circles = cv2.HoughCircles(
        gray,
        cv2.HOUGH_GRADIENT,
        dp=1.2,
        minDist=50,
        param1=param1,
        param2=param2,
        minRadius=15,
        maxRadius=175
    )


    detected_count = 0

    if circles is not None:

        circles = np.uint16(np.around(circles))


        # IMPROVEMENT 3: Reject False Circles

        # IMPROVEMENT 4: Stability Across Frames

        best_circle = None
        best_dist   = float('inf')

        for i in circles[0, :]:
            x, y, r = int(i[0]), int(i[1]), int(i[2])

            # IMPROVEMENT 3: Skip circles outside the valid size range
            if r < 20 or r > 200:
                continue  # Reject – too small (noise) or too large (face/background)

            # IMPROVEMENT 4: Calculate distance from this circle
            # to the previously detected circle
            if prev_circle is not None:
                dist = (x - prev_circle[0]) ** 2 + (y - prev_circle[1]) ** 2
            else:

                dist = 0

            # Keep whichever valid circle is closest to the last known position
            if dist < best_dist:
                best_dist   = dist
                best_circle = (x, y, r)

        #drawing
        if best_circle is not None:
            x, y, r = best_circle


            prev_circle = (x, y)


            detected_count = 1


            cv2.circle(frame, (x, y), r, (0, 255, 0), 2)


            cv2.circle(frame, (x, y), 2, (0, 0, 255), 3)


            cv2.putText(frame, f"R={r}", (x, y - 10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 2)


            cv2.putText(frame, f"({x},{y})", (x, y + r + 18),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.45, (0, 255, 255), 1)

    else:
        # ENHANCEMENT 3: Show a warning when no circle is found

        prev_circle = None
        cv2.putText(frame, "No circle detected", (20, 40),
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)



    # ENHANCEMENT 4: Show the number of accepted circles this frame
    cv2.putText(frame, f"Circles: {detected_count}", (20, 80),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 0), 2)

    # ENHANCEMENT 5: Calculate and display live FPS
    elapsed = time.time() - start_time
    fps = frame_count / elapsed
    cv2.putText(frame, f"FPS: {int(fps)}", (20, 110),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 0), 2)

    # ── Show the final annotated frame ──
    cv2.imshow("Circle Detection", frame)


    key = cv2.waitKey(1)


    if key == ord('q'):
        break


    elif key == ord('s'):
        cv2.imwrite("screenshot.png", frame)
        print("Screenshot saved as screenshot.png")


cap.release()
cv2.destroyAllWindows()